#### 🔷**Noisy Quantum Simulation via Matrix Product Density Operators (MPDO)**

This notebook implements and analyzes noisy quantum simulations for a random quantum circuit using the **Matrix Product Density Operator (MPDO)** approach. 

We evaluate the performance of the MPDO simulator across three distinct noise channels:
* **Dephasing Channel**
* **Depolarizing Channel**
* **Amplitude Damping Channel**

##### **Key Objectives:**
1. **Parameter Sweep:** Benchmark the simulation by varying the maximum bond dimension ($\chi_{\max}$) and the maximum Kraus rank ($\kappa_{\max}$).
2. **Validation:** Verify the accuracy of the MPDO approach by calculating its **fidelity** against an exact noisy simulation conducted in `Qiskit`.

In [2]:
import sys, os, re, glob
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.quantum_info import DensityMatrix, Kraus
from qiskit.circuit.library import UGate, CXGate, CZGate
sys.path.insert(0, os.path.abspath("../.."))  
from src.mpdo import MPDOState, Dephasing, AmplitudeDamping, Depolarizing
import matplotlib.pyplot as plt

#### ***Random one and two qubit gate: `random_one_qgate` & `random_two_qgate`***

In [3]:
def random_one_qgate(rng: np.random.Generator) -> np.ndarray:
    """
    Draw (theta, phi, lam) and return Qiskit's U gate matrix.
    build_circuit uses the same angles via qc.u(theta, phi, lam).
    """
    theta, phi, lam = rng.uniform(0, 2 * np.pi, size=3)
    return UGate(theta, phi, lam).to_matrix()          # (2, 2) complex


In [4]:
def random_two_qgate(rng: np.random.Generator) -> np.ndarray:
    """
    Choose CNOT or CZ with equal probability.
    build_circuit uses the same draw via qc.cx / qc.cz.
    """
    if rng.random() < 0.5:
        return CXGate().to_matrix()                    # (4, 4) complex
    else:
        return CZGate().to_matrix()

In [5]:
def apply_layer(state, single_gates, pairs, two_qubit_gates, channel):
    # Step 1: single-qubit gates
    for i in range(state.N):
        state.apply_single_qubit_gate(i, single_gates[i])

    # Step 2: two-qubit gates + noise
    for (k, k1), U in zip(pairs, two_qubit_gates):
        state.apply_two_qubit_gate(k, k1, U)
        state.apply_noise(k,  channel)
        state.apply_noise(k1,  channel)

    # Step 3: canonicalize (chi_max handled internally)
    state.canonicalize()


#### ***Apply layer and run `simulate_circuit`***

In [6]:
#  simulate_circuit: creates the state, runs all layers
def simulate_circuit(N, D, epsilon, chi_max, kappa_max, seed=0):
    rng = np.random.default_rng(seed)
    state = MPDOState.init_product_state(N,chi_max, kappa_max)
    channel = AmplitudeDamping(epsilon)

    for layer in range(D):
        print(f"  Layer {layer+1}/{D}  |  {state}")   # MPDOState.__repr__

        single_gates = [random_one_qgate(rng) for _ in range(N)]

        # Determine alternating structural layers
        pairs = ([(k, k + 1) for k in range(0, N - 1, 2)] if layer %2 ==0
                 else
                 [(k, k+1) for k in range(1, N-1, 2)]
                 )
        # Sample two-qubit operators for each pair configuration
        tq_gates = [random_two_qgate(rng) for _ in pairs]

        apply_layer(state, single_gates, pairs, tq_gates, channel)  
    
    return state 
        


### ***Fidality calculation***

In [7]:
def matrix_sqrt(M: np.ndarray) -> np.ndarray:
    """Matrix square root of a Hermitian PSD matrix."""
    eigenvalues, V = np.linalg.eigh(M)
    eigenvalues    = np.maximum(eigenvalues, 0.0)
    return V @ np.diag(np.sqrt(eigenvalues)) @ V.conj().T


def fidelity(rho: np.ndarray, sigma: np.ndarray) -> float:
    """
    F(rho, sigma) = Tr( sqrt( sqrt(rho) @ sigma @ sqrt(rho) ) )
    """
    sqrt_rho      = matrix_sqrt(rho)
    M             = sqrt_rho @ sigma @ sqrt_rho
    eigenvalues_M, _ = np.linalg.eigh(M)
    eigenvalues_M = np.maximum(eigenvalues_M, 0.0)
    F             = np.sum(np.sqrt(eigenvalues_M))
    return float(np.clip(np.real(F), 0.0, 1.0))

### ***Qiskit EXACT SIMULATION***

In [8]:
def build_circuit(N: int, D: int, seed: int) -> QuantumCircuit:
    """
    Build the 1D random circuit from Figure 3.
    RNG draw order matches simulate_circuit() exactly for the same seed.
    """

    rng = np.random.default_rng(seed)
    qc = QuantumCircuit(N)

    for layer in range(D):
        # Apply single-qubit gate layer
        for i in range(N):
            theta, phi, lam = rng.uniform(0, 2 * np.pi, size=3)
            qc.u(theta, phi, lam, i)

        # Determine alternating 1D nearest-neighbor pairs
        pairs = (
            [(k, k + 1) for k in range(0, N - 1, 2)]
            if layer % 2 == 0 else
            [(k, k + 1) for k in range(1, N - 1, 2)]
        )

        # Apply two-qubit gate layer
        for q1, q2 in pairs:
            if rng.random() < 0.5:
                qc.cx(q1, q2)
            else:
                qc.cz(q1, q2)

    return qc

#### ***Build noise model for `Qiskit` exact simulation***

In [9]:
def build_noise_model(model_name: str, rate: float) -> NoiseModel:
    """
    E⊗E noise after each two-qubit gate, matching equation (2.8).
    """
    noise_model = NoiseModel()

    # if model_name == "depolarizing":
    #     err_1q = depolarizing_error(rate, 1)
    #     err_2q = err_1q.expand(err_1q)
    #     noise_model.add_all_qubit_quantum_error(err_2q, ["cx", "cz"])

    if model_name == "depolarizing":
        p = rate
        K0 = np.sqrt(1.0 - 3*p/4) * np.eye(2, dtype=complex)
        K1 = np.sqrt(p/4) * np.array([[0,  1 ], [1,  0 ]], dtype=complex)  
        K2 = np.sqrt(p/4) * np.array([[0, -1j], [1j, 0 ]], dtype=complex)  
        K3 = np.sqrt(p/4) * np.array([[1,  0 ], [0, -1 ]], dtype=complex)  
        kraus_2q = [np.kron(a, b) for a in [K0,K1,K2,K3] for b in [K0,K1,K2,K3]]
        noise_model.add_all_qubit_quantum_error(Kraus(kraus_2q), ["cx", "cz"])

    elif model_name == "dephasing":
        p  = rate
        K0 = np.sqrt(1 - p) * np.eye(2, dtype=complex)
        K1 = np.sqrt(p) * np.array([[1, 0], [0, -1]], dtype=complex)
        kraus_2q = [np.kron(a, b) for a in [K0, K1] for b in [K0, K1]]
        noise_model.add_all_qubit_quantum_error(Kraus(kraus_2q), ["cx", "cz"])

    elif model_name == "amplitude_damping":
        g  = rate
        A0 = np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=complex)
        A1 = np.array([[0, np.sqrt(g)], [0, 0]],     dtype=complex)
        kraus_2q = [np.kron(a, b) for a in [A0, A1] for b in [A0, A1]]
        noise_model.add_all_qubit_quantum_error(Kraus(kraus_2q), ["cx", "cz"])

    else:
        raise ValueError(f"Unknown noise model: '{model_name}'.")

    return noise_model

In [10]:
def simulate_exact(
    N: int, D: int, model_name: str, rate: float, seed: int 
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns rho0 (noiseless) and rho_e (noisy) as numpy arrays.
    """
    qc = build_circuit(N, D, seed=seed)

    # noiseless
    qc_nl = qc.copy()
    qc_nl.save_density_matrix()
    rho0 = np.array(
        AerSimulator().run(qc_nl).result().data()["density_matrix"]
    )

    # noisy
    qc_ns = qc.copy()
    qc_ns.save_density_matrix()
    noise_model   = build_noise_model(model_name, rate)
    backend_noisy = AerSimulator(noise_model=noise_model)
    rho_e = np.array(
        backend_noisy.run(qc_ns).result().data()["density_matrix"]
    )

    return rho0, rho_e

### ***Circuit Parameters***

In [11]:
# Simulation Configuration Parameters
NUM_QUBITS = 8
CIRCUIT_DEPTH = 20
NOISE_MODEL = "amplitude_damping"
NOISE_RATE = 10**(-10)    
BOND_MAX_CHI_LIST =  32 #np.arange(1, 17, 1)       # Maximum virtual MPS/MPDO rank
BOND_MAX_KAPPA = np.arange(1, 17, 1)               # Maximum auxiliary environment dimension
RANDOM_SEED = 42

##### ***Exact qiskit simulation***

In [12]:
rho0, rho_qiskit = simulate_exact(NUM_QUBITS, CIRCUIT_DEPTH, NOISE_MODEL, NOISE_RATE, seed=RANDOM_SEED)

##### ***MPDO simulation***

In [13]:
filename = f"./inner_dim/fidelity_NEW_N{NUM_QUBITS}_D{CIRCUIT_DEPTH}_noise{NOISE_RATE}_seed{RANDOM_SEED}.txt"
# filename = f"./chi_dim/fidelity_NEW_N{NUM_QUBITS}_D{CIRCUIT_DEPTH}_noise{NOISE_RATE}_seed{RANDOM_SEED}.txt"

with open(filename, mode="w") as file:

    file.write("BOND_MAX_KAPPA\tfidelity\n")
    # file.write("BOND_MAX_CHI\tfidelity\n")

    for kappa in BOND_MAX_KAPPA:
        # kappa = 2 * chi
        chi = BOND_MAX_CHI_LIST #2  * kappa

        state = simulate_circuit(
            N=NUM_QUBITS,
            D=CIRCUIT_DEPTH,
            epsilon=NOISE_RATE,
            # chi_max=chi,
            chi_max = chi,
            kappa_max=kappa,
            seed=RANDOM_SEED,
        )

        rho_mpdo = state.to_density_matrix()
        fidelity_score = fidelity(rho_mpdo, rho_qiskit)

        file.write(f"{kappa}\t{fidelity_score}\n")
        # file.write(f"{chi}\t{fidelity_score}\n")

print(f"Success! Data saved to: {filename}")

  Layer 1/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 2/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 3/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 4/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 5/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 6/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 7/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 8/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 9/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 10/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 11/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 12/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 13/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 14/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 15/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 16/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 17/20  |  MPDOState(N=8, chi_max=32, kappa_max=1)
  Layer 18/20  |  MPDOS

In [14]:
print(state)           
print(state.N)         
print(state.bond_dims) 
print(len(state))

MPDOState(N=8, chi_max=32, kappa_max=16)
8
[1, 32, 32, 32, 32, 32, 32, 32, 1]
8
